# Active learning with STACNotator

This notebook walks through one full active-learning round against a STACNotator campaign:

1. Log in via the browser and open a campaign
2. Fetch the labeled samples and split them into train/test
3. Train a deliberately simple model (lat/lon as the only features)
4. Predict over a pixel grid covering the campaign extent and write the result as a COG
5. Register the prediction as a map layer with a class legend, so annotators see where the model is confused
6. After annotators add labels, grow the training set with `update_samples` and retrain

The lat/lon model is a stand-in: it only demonstrates the data plumbing. In a real setup you
would featurize with raster bands or embeddings; loading raster data through the SDK is on the
roadmap (see the README TODOs).

In [4]:
# The SDK is not on PyPI yet; install it from this repo checkout
# (this notebook lives in sdk/examples/, the package root sdk/ is one dir up).
%pip install -q .. scikit-learn shapely rasterio matplotlib

Note: you may need to restart the kernel to use updated packages.


## 1. Login and pick a campaign

`login` opens your browser once and caches the credential locally. Against a local dev
backend (`AUTH_PROVIDER=local`) it just works without a browser.

In [2]:
import stacnotator as snt

URL = "http://localhost:5173"  # your STACNotator app URL

snt.login(URL)
snt.campaigns()

pci id for fd 5: 10de:28a0, driver (null)
pci id for fd 6: 10de:28a0, driver (null)
pci id for fd 7: 10de:28a0, driver (null)
pci id for fd 6: 10de:28a0, driver (null)
pci id for fd 7: 10de:28a0, driver (null)


Opening in existing browser session.


,id,name,created_at,is_admin,is_member,is_public
0,91,venezuela earthquake,2026-07-01T15:38:34.085078,True,True,False
1,83,test-landsat-viz,2026-06-08T13:46:18.466163,True,True,False
2,82,Bungoma_cropland,2026-06-08T07:08:49.798690,True,True,False
3,81,BR_Pasture_subStrata_n1575_2000-2025,2026-06-05T19:04:24.763447,True,True,False
4,80,Ukraine_WinterCrops_20260605,2026-06-05T13:53:22.961535,True,True,False
5,74,BR-Pasture_subStrata_2000-2025,2026-06-02T17:57:44.630379,True,True,False
6,71,Programa Escuelas Rurales 2026,2026-06-02T09:47:01.012571,True,True,False
7,69,"Kenya, Muranga County - Cropland 2025-2026",2026-05-21T05:47:37.850658,True,True,False
8,66,Maryland Solar Farm Validation (BWM 2026 UMDGC...,2026-05-14T15:01:17.174609,True,True,False
9,63,"Kenya, Bungoma County - Cropland 2025-2026",2026-05-06T22:10:52.345540,True,True,False


In [ ]:
CAMPAIGN_ID = 91  # pick one from the table above

campaign = snt.campaign(CAMPAIGN_ID)
print(campaign)
print("labels:", campaign.labels)
print("extent:", campaign.extent)

## 2. Fetch samples and split

One row per labeled annotation. The test set is held out once and stays fixed for the whole
campaign, so scores are comparable across rounds.

In [ ]:
from sklearn.model_selection import train_test_split

samples = campaign.get_samples()
print(f"{len(samples)} labeled samples")
samples.head()

In [ ]:
train, test = train_test_split(samples, test_size=0.2, random_state=42)
len(train), len(test)

## 3. Train a very simple model

The only features are lon/lat. The SDK hands geometries over as they are, so reducing
them to coordinates is our job here: point samples already carry lat/lon, polygon and box
samples are reduced to their centroid.

In [ ]:
import numpy as np
from shapely.geometry import shape
from sklearn.ensemble import RandomForestClassifier


def featurize(df):
    centroids = df["geometry"].map(lambda g: shape(g).centroid)
    lon = df["lon"].fillna(centroids.map(lambda c: c.x))
    lat = df["lat"].fillna(centroids.map(lambda c: c.y))
    return np.column_stack([lon, lat]), df["label_id"].to_numpy(dtype="int64")


model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(*featurize(train))
print(f"held-out accuracy: {model.score(*featurize(test)):.2f}")

## 4. Predict a dense class map over the campaign extent

The model is evaluated on every pixel of a regular grid covering the extent, giving a
dense raster you can upload as a map. `PIXEL_SIZE_DEG` sets the resolution; coarser
pixels mean a smaller file and faster prediction.

In [ ]:
import numpy as np

PIXEL_SIZE_DEG = 0.05  # make this smaller for a finer map

west, south, east, north = campaign.extent
width = max(2, int(np.ceil((east - west) / PIXEL_SIZE_DEG)))
height = max(2, int(np.ceil((north - south) / PIXEL_SIZE_DEG)))

lons = np.linspace(west, east, width)
lats = np.linspace(north, south, height)  # north up: first row is the top
grid = np.column_stack([np.tile(lons, height), np.repeat(lats, width)])

class_raster = model.predict(grid).reshape(height, width).astype("uint8")
print(f"{width} x {height} pixels, classes {np.unique(class_raster)}")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

class_ids = sorted(campaign.labels)
cmap = ListedColormap(plt.get_cmap("tab10").colors[: len(class_ids)])

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(class_raster, extent=(west, east, south, north), cmap=cmap, interpolation="nearest")
colorbar = fig.colorbar(im, ticks=class_ids)
colorbar.ax.set_yticklabels([campaign.labels[i] for i in class_ids])
ax.set_title("Predicted class map over the campaign extent")
plt.show()

## 5. Write the prediction as a COG and register it

The tiler fetches the file by URL, so after writing it locally, upload it anywhere the tiler
can reach (an Azure blob with a SAS token, an S3 presigned URL, any public file host) and put
that URL in `COG_URL`.

In [ ]:
import rasterio
from rasterio.transform import from_bounds

with rasterio.open(
    "predictions.tif",
    "w",
    driver="COG",
    width=width,
    height=height,
    count=1,
    dtype="uint8",
    crs="EPSG:4326",
    transform=from_bounds(west, south, east, north, width, height),
) as dst:
    dst.write(class_raster, 1)

print("wrote predictions.tif; upload it and set COG_URL below")

In [ ]:
COG_URL = "https://<your-storage>/predictions.tif?<sas-token>" 

layer = campaign.register_pred_layer(
    COG_URL,
    classes=campaign.labels,  # class values match label ids, legend shows label names
)
layer

In [ ]:
# Registration runs asynchronously on the server; re-run until status is "ready".
campaign.pred_layers()

Annotators now see the prediction overlay with its legend in the annotation UI and can add
labels where the model is wrong.

## 6. Next round: grow the training set and retrain

`update_samples` appends only samples that are new since the last fetch. The held-out test
set is passed as `exclude`, so it never leaks into training.

In [ ]:
before = len(train)
train = campaign.update_samples(train, exclude=test)
print(f"training set: {before} -> {len(train)} samples")

model.fit(*featurize(train))
print(f"held-out accuracy: {model.score(*featurize(test)):.2f}")

From here the loop repeats: predict over the extent, upload, register the next layer (names
auto-number as `prediction-2`, `prediction-3`, ...), wait for annotators, update, retrain.